# 02 — LoRA Fine-Tuning
Train a LoRA adapter on FOLIO using the config-driven `scripts/train.py` approach.
Config is loaded from `configs/lora/phi35.yml` — change this to switch models.

In [1]:
# ── Colab setup ────────────────────────────────────────────────────────────
import os
from google.colab import drive, userdata   # ← add userdata here
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    # Update this path if your outputs folder is elsewhere
    OUTPUT_BASE = '/content/drive/MyDrive/DL/project/slm_logic_hardening-main/outputs'
    !pip install -q transformers datasets pyyaml sentencepiece
    # Navigate directly to your existing project folder in Drive
    %cd '/content/drive/MyDrive/DL/project/slm_logic_hardening-main'
else:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path().resolve()))
    OUTPUT_BASE = 'outputs'

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

Mounted at /content/drive
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1988/3436745669.py", line 13, in <cell line: 0>
    get_ipython().run_line_magic('cd', "'/content/drive/MyDrive/DL/project/slm_logic_hardening-main'")
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exce

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
device = 'mps' if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 1. Load config

In [ ]:
import yaml

#CONFIG_PATH = 'configs/lora/phi35.yml'
CONFIG_PATH = 'configs/lora/phi35_folio_pw.yml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('Model key:', cfg['model_key'])
print('Experiment:', cfg['experiment_name'])
print('LoRA rank:', cfg['lora']['r'])
print('Learning rate:', cfg['training']['learning_rate'])
print('Epochs:', cfg['training']['num_train_epochs'])
print(cfg.get('dataset_sample_sizes'))  # should show {'proofwriter': 2000}

Model key: phi35
Experiment: phi35_lora_folio_pw
LoRA rank: 64
Learning rate: 0.0001
Epochs: 5
{'proofwriter': 2000}


## 2. Load model and apply LoRA

In [ ]:
from models.model_loader import load_model
from peft import LoraConfig, get_peft_model

model, tokenizer, device = load_model(cfg['model_key'], mode='baseline')

lc = cfg['lora']
lora_config = LoraConfig(
    r=lc['r'],
    lora_alpha=lc['lora_alpha'],
    target_modules=lc['target_modules'],
    lora_dropout=lc['lora_dropout'],
    bias=lc['bias'],
    task_type=lc['task_type'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282


## 3. Prepare datasets

In [ ]:
from data.load_data import load_datasets_for_training
from data.preprocess import DATASET_PREPROCESSORS
from scripts.train import build_tokenize_fn
from datasets import concatenate_datasets

dataset_names = cfg.get('datasets', ['folio'])
sample_sizes = cfg.get('dataset_sample_sizes', {})
raw_datasets = load_datasets_for_training(dataset_names, sample_sizes=sample_sizes)

max_length = cfg['training']['max_length']
tokenize_fn = build_tokenize_fn(tokenizer, max_length)
columns_to_keep = ['input_ids', 'attention_mask', 'labels']

all_train = []
folio_val = None  # validate on FOLIO only

for name, raw in raw_datasets.items():
    preprocessor = DATASET_PREPROCESSORS[name]
    processed = preprocessor(raw)
    t = processed['train'].map(tokenize_fn, load_from_cache_file=False)
    t = t.remove_columns([c for c in t.column_names if c not in columns_to_keep])
    all_train.append(t)
    if name == 'folio' and 'validation' in processed:
        v = processed['validation'].map(tokenize_fn, load_from_cache_file=False)
        v = v.remove_columns([c for c in v.column_names if c not in columns_to_keep])
        folio_val = v

train_dataset = concatenate_datasets(all_train) if len(all_train) > 1 else all_train[0]
val_dataset = folio_val

print(f'Datasets: {dataset_names}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} (FOLIO only)')


Loading dataset: folio
Loading dataset: proofwriter


Map: 100%|##########| 1001/1001 [00:00<?, ? examples/s]

Map: 100%|##########| 203/203 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map: 100%|##########| 203/203 [00:00<?, ? examples/s]

Map: 100%|##########| 1078/1078 [00:00<?, ? examples/s]

Map: 100%|##########| 92796/92796 [00:00<?, ? examples/s]

Map: 100%|##########| 46398/46398 [00:00<?, ? examples/s]

Map: 100%|##########| 1078/1078 [00:00<?, ? examples/s]

Datasets: ['folio', 'proofwriter']
Train: 2079 | Val: 203 (FOLIO only)


## 4. Train

In [ ]:
from pathlib import Path
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

tc = cfg['training']
output_dir = Path(OUTPUT_BASE) / cfg['model_key'] / cfg['experiment_name']
output_dir.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=tc['num_train_epochs'],
    per_device_train_batch_size=tc['per_device_train_batch_size'],
    per_device_eval_batch_size=tc['per_device_eval_batch_size'],
    learning_rate=tc['learning_rate'],
    warmup_steps=tc['warmup_steps'],
    max_grad_norm=tc['max_grad_norm'],
    logging_steps=tc['logging_steps'],
    eval_strategy=tc['eval_strategy'],
    save_strategy=tc['save_strategy'],
    fp16=tc['fp16'],
    bf16=tc['bf16'],
    load_best_model_at_end=tc['load_best_model_at_end'],
    metric_for_best_model=tc['metric_for_best_model'],
    report_to=tc['report_to'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,4.592041,4.959445
2,4.657838,4.978423
3,4.639806,5.056008


TrainOutput(global_step=1560, training_loss=4.738804098275992, metrics={'train_runtime': 387.9225, 'train_samples_per_second': 26.797, 'train_steps_per_second': 6.702, 'total_flos': 5.367445515848909e+16, 'train_loss': 4.738804098275992, 'epoch': 3.0})

## 5. Save adapter

In [ ]:
adapter_path = output_dir / 'final_adapter'
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print(f'Adapter saved to {adapter_path}')

Adapter saved to /content/drive/MyDrive/slm_logic_hardening/outputs/phi35/phi35_lora_folio_pw/final_adapter
